In [1]:
import sys
sys.path.append("..")

from src.data_loader import load_clean_data, get_features_and_target

df_clean = load_clean_data()
X, y = get_features_and_target(df_clean)

In [2]:
X.shape, y.shape


((5758, 8), (5758,))

In [4]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [5]:
# Define all the models
models = {
    'kNN': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=24))]),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(random_state=42))]),
    'XGBoost': Pipeline([('scaler', StandardScaler()), ('clf', XGBClassifier(random_state=42))]),
}

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1', 'roc_auc']

results = {}
for name, pipe in models.items():
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    results[name] = scores

In [9]:
results.items()

dict_items([('kNN', {'fit_time': array([0.00679517, 0.00284863, 0.        , 0.00207114, 0.01466966,
       0.01199579, 0.01732326, 0.0166986 , 0.0167737 , 0.01545358]), 'score_time': array([0.04611611, 0.04559565, 0.04713488, 0.05011773, 0.04404402,
       0.03023219, 0.03123593, 0.03327632, 0.03336859, 0.03687191]), 'test_accuracy': array([0.68229167, 0.67881944, 0.75520833, 0.71006944, 0.68923611,
       0.73784722, 0.71180556, 0.71354167, 0.67826087, 0.7426087 ]), 'test_f1': array([0.72481203, 0.72592593, 0.78603945, 0.74268105, 0.73481481,
       0.76946565, 0.74303406, 0.7480916 , 0.72180451, 0.77844311]), 'test_roc_auc': array([0.7477107 , 0.75481459, 0.80931045, 0.7676494 , 0.75129635,
       0.77983451, 0.77160895, 0.77222801, 0.75193846, 0.79369559])}), ('Logistic Regression', {'fit_time': array([0.164294  , 0.02009606, 0.01639891, 0.01735973, 0.01650524,
       0.0137887 , 0.01703   , 0.01675534, 0.01910663, 0.02962589]), 'score_time': array([0.01694489, 0.0130806 , 0.0166361

In [8]:
# Compiling the results

summary = pd.DataFrame({
    name: {
        'accuracy_mean': res['test_accuracy'].mean(),
        'accuracy_std': res['test_accuracy'].std(),
        'f1_mean': res['test_f1'].mean(),
        'f1_std': res['test_f1'].std(),
        'roc_auc_mean': res['test_roc_auc'].mean(),
        'roc_auc_std': res['test_roc_auc'].std(),
    }
    for name, res in results.items()
}).T

summary

,accuracy_mean,accuracy_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
kNN,0.709969,0.026602,0.747511,0.021841,0.770009,0.019026
Logistic Regression,0.652308,0.021767,0.709821,0.020633,0.713037,0.020902
Random Forest,0.748177,0.010085,0.778905,0.009837,0.820273,0.015025
XGBoost,0.737928,0.013331,0.773531,0.012425,0.805089,0.016508


In [11]:
# HYperparameter tuning for XGBoost
from sklearn.model_selection import RandomizedSearchCV

pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', XGBClassifier(random_state=42, eval_metric='logloss'))
])

param_dist = {
    'clf__n_estimators': [50, 100, 150, 200, 300, 500],
    'clf__max_depth': [2, 3, 4, 6, 8],
    'clf__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2, 0.3],
    'clf__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'clf__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'clf__reg_alpha': [0, 0.01, 0.1, 1, 5],
    'clf__reg_lambda': [0.1, 1, 5, 10],
}

search = RandomizedSearchCV(
    pipe_xgb,             # your existing XGBoost Pipeline
    param_distributions=param_dist,
    n_iter=25,
    scoring='f1',
    cv=cv,                 # same StratifiedKFold as before
    random_state=42,
    n_jobs=-1,
    verbose=2       # print progress as it goes
)
search.fit(X, y)

Fitting 10 folds for each of 25 candidates, totalling 250 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'clf__colsample_bytree': [0.6, 0.7, ...], 'clf__learning_rate': [0.01, 0.03, ...], 'clf__max_depth': [2, 3, ...], 'clf__n_estimators': [50, 100, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",25
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best est

In [12]:
search.best_params_

{'clf__subsample': 0.8,
 'clf__reg_lambda': 5,
 'clf__reg_alpha': 0.01,
 'clf__n_estimators': 500,
 'clf__max_depth': 8,
 'clf__learning_rate': 0.3,
 'clf__colsample_bytree': 0.9}

In [13]:
search.best_score_

np.float64(0.7789379498170482)

In [ ]:
from sklearn.model_selection import cross_validate

best_xgb = search.best_estimator_
tuned_scores = cross_validate(best_xgb, X, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'])

In [15]:
print('Accuracy:', tuned_scores['test_accuracy'].mean(), '±', tuned_scores['test_accuracy'].std())
print('F1:      ', tuned_scores['test_f1'].mean(), '±', tuned_scores['test_f1'].std())
print('ROC-AUC: ', tuned_scores['test_roc_auc'].mean(), '±', tuned_scores['test_roc_auc'].std())

Accuracy: 0.7429652777777778 ± 0.01802912548263335
F1:       0.7753912427402947 ± 0.014850256653173173
ROC-AUC:  0.8059006794020499 ± 0.017618407559678046


In [16]:
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=42))
])
final_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['NumValenceElectrons','qed','TPSA',...,'BertzCT','MolWt','MolLogP']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [18]:
from pathlib import Path
import joblib
Path('../models').mkdir(exist_ok=True) # create the folder if not existed

joblib.dump(final_model, '../models/best_classifier.pkl')

['../models/best_classifier.pkl']

In [19]:
import json

with open('../models/descriptor_columns.json', 'w') as f:
    json.dump(list(X.columns), f)